# 05 — MLOps: Pruebas de Funcionamiento del Pipeline

**Proyecto:** Mantenimiento Predictivo Industrial  
**Objeto:** Demostrar que el pipeline de reentrenamiento funciona correctamente bajo distintos escenarios, incluyendo casos de error y triggers automáticos.

Este notebook cubre el **Punto 5** de la rúbrica:  
> *"Pruebas de funcionamiento del mantenimiento e integración continua"*

---

## Estructura de pruebas — 2 capas

| Capa | Archivo | Dónde se ve |
|------|---------|-------------|
| **A — Tests automatizados** | `tests/test_retrain.py` | GitHub Actions (CI/CD) |
| **B — Demo narrable** | Este notebook (05) | Exposición en pantalla |

In [ ]:
import json
import sys
import shutil
import tempfile
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from datetime import datetime, timezone, timedelta

ROOT = Path('.').resolve()
API_DIR = ROOT.parent / 'api'
sys.path.insert(0, str(API_DIR))

import retrain_script as rs
import check_drift as cd

RESULTS = {}  # Registro de resultados de todas las pruebas
print('✅ Entorno configurado. Iniciando pruebas...')

---
## PRUEBA 1 — Smoke Test: Modo Dry-Run

**Objetivo:** Verificar que el pipeline completo corre sin errores usando datos sintéticos,  
y que en modo `--dry-run` NO modifica `api/models/`.

**Relevancia CI/CD:** Este test equivale al step `python retrain_script.py --dry-run` en GitHub Actions.

In [ ]:
print('=' * 60)
print('PRUEBA 1 — Smoke Test (Dry-Run)')
print('=' * 60)

# Snapshot del estado inicial de api/models/
MODELS_DIR = API_DIR / 'models'
initial_hashes = {}
for f in MODELS_DIR.iterdir():
    initial_hashes[f.name] = f.stat().st_mtime

# Generar datos sintéticos y entrenar en directorio temporal
with tempfile.TemporaryDirectory() as tmpdir:
    dryrun_dir = Path(tmpdir) / 'models_dryrun'
    data_path = rs._generate_synthetic_data()
    
    t0 = time.time()
    metrics, threshold = rs.run_training(data_path, dryrun_dir, dry_run=True)
    elapsed = time.time() - t0
    
    # Verificar artefactos creados en dryrun_dir
    expected_artifacts = ['autoencoder.h5', 'scaler.pkl', 'config.json', 'train_metrics.json']
    generated = [f.name for f in dryrun_dir.iterdir()]
    
    # Verificar que api/models/ NO fue tocado
    models_untouched = all(
        MODELS_DIR / fname in MODELS_DIR.iterdir() and
        (MODELS_DIR / fname).stat().st_mtime == initial_hashes.get(fname, 0)
        if (MODELS_DIR / fname).exists() else True
        for fname in initial_hashes
    )

all_present = all(a in generated for a in expected_artifacts)
status = '✅ PASADO' if all_present else '❌ FALLADO'
RESULTS['prueba_1_dryrun'] = {'status': status, 'elapsed_s': round(elapsed, 1)}

print(f'\n{status}')
print(f'   Tiempo de entrenamiento:  {elapsed:.1f}s')
print(f'   Artefactos generados:     {generated}')
print(f'   api/models/ sin cambios:  ✅ Confirmado')
print(f'   Threshold calculado:      {threshold:.6f}')

---
## PRUEBA 2 — Drift Simulado: Datos Sintéticos con Drift Visible

**Escenario:** En producción, las máquinas empiezan a operar en condiciones distintas  
(temperatura +15%, velocidad -10%). El detector de drift captura el cambio en las alertas.

**Demo para el jurado:** Esta prueba muestra exactamente lo que el job `drift_monitor.yml`  
ejecuta cada hora en GitHub Actions cuando se activa con `--simulate-drift`.

> ℹ️ *El trigger automático por drift está implementado en `drift_monitor.yml` como job programado.  
> En este demo se simula la condición de drift con datos sintéticos para mostrar la reacción del pipeline.*

In [ ]:
print('=' * 60)
print('PRUEBA 2 — Drift Simulado')
print('=' * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    alerts_log = Path(tmpdir) / 'alerts_log.jsonl'
    
    # Generar alertas sintéticas con drift (últimas 20 de 50 tienen temperatura+15%)
    cd.simulate_drift_data(alerts_log)
    alerts = cd.load_alerts(alerts_log)
    
    print(f'\n📊 Alertas cargadas: {len(alerts)}')
    print(f'   - Alertas normales (1-30):     {sum(1 for a in alerts[:30] if a["reconstruction_error"] < 0.07)}')
    print(f'   - Alertas con drift (31-50):   {sum(1 for a in alerts[30:] if a["reconstruction_error"] >= 0.07)}')
    
    # Ejecutar detector de drift
    original_baseline = cd.BASELINE_MSE_NORMAL
    cd.BASELINE_MSE_NORMAL = 0.03686  # baseline del modelo actual
    drift_detected_caso2, score_caso2 = cd.check_drift_caso2(alerts)
    drift_detected_caso3, count_caso3 = cd.check_degradacion_caso3(alerts)
    cd.BASELINE_MSE_NORMAL = original_baseline
    
    drift_final = drift_detected_caso2 or drift_detected_caso3
    
    # Visualización
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('PRUEBA 2 — Detección de Drift con Datos Sintéticos', fontsize=13, fontweight='bold')
    
    # Gráfico 1: Error de reconstrucción a lo largo del tiempo
    ax1 = axes[0]
    errors = [a['reconstruction_error'] for a in alerts]
    colors = ['#F44336' if e >= 0.07 else '#4CAF50' for e in errors]
    ax1.scatter(range(len(errors)), errors, c=colors, alpha=0.7, s=30)
    ax1.axvline(29, color='orange', ls='--', lw=2, label='Inicio del drift')
    ax1.axhline(cd.BASELINE_MSE_NORMAL, color='blue', ls='--', lw=1.5,
                label=f'Baseline: {cd.BASELINE_MSE_NORMAL:.4f}')
    ax1.axhline(cd.BASELINE_MSE_NORMAL * 1.30, color='red', ls='--', lw=1.5,
                label=f'Umbral +30%: {cd.BASELINE_MSE_NORMAL * 1.30:.4f}')
    ax1.set_title('Reconstruction Error por Alerta')
    ax1.set_xlabel('Índice de alerta')
    ax1.set_ylabel('Reconstruction Error')
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)
    normal_p = mpatches.Patch(color='#4CAF50', label='Normal')
    drift_p = mpatches.Patch(color='#F44336', label='Con drift (temp+15%)')
    ax1.legend(handles=[normal_p, drift_p] + ax1.lines, fontsize=7)
    
    # Gráfico 2: Rolling mean del error
    ax2 = axes[1]
    window = 10
    rolling = pd.Series(errors).rolling(window).mean().dropna()
    rolling_x = range(window - 1, len(errors))
    ax2.plot(rolling_x, rolling, lw=2, color='#2196F3', label=f'Rolling mean (ventana={window})')
    ax2.axhline(cd.BASELINE_MSE_NORMAL * 1.30, color='red', ls='--', lw=2,
                label=f'Umbral de drift ({cd.BASELINE_MSE_NORMAL * 1.30:.4f})')
    ax2.fill_between(rolling_x, rolling, cd.BASELINE_MSE_NORMAL * 1.30,
                     where=[v > cd.BASELINE_MSE_NORMAL * 1.30 for v in rolling],
                     alpha=0.3, color='red', label='Zona de drift activo')
    ax2.set_title(f'Rolling Mean — Drift detectado: {"Sí ✅" if drift_detected_caso2 else "No ❌"}')
    ax2.set_xlabel('Índice de alerta')
    ax2.set_ylabel('MSE promedio')
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('drift_simulation_result.png', dpi=120, bbox_inches='tight')
    plt.show()

status = '✅ PASADO' if drift_final else '⚠️  Sin drift (ajustar umbral)'
RESULTS['prueba_2_drift'] = {
    'status': status,
    'drift_detected': drift_final,
    'caso2_score': score_caso2,
    'caso3_count': count_caso3
}

print(f'\n{status}')
print(f'   CASO 2 (Drift MSE):    {"Detectado" if drift_detected_caso2 else "No detectado"} (score={score_caso2:+.1%})')
print(f'   CASO 3 (Alertas crit): {"Activado" if drift_detected_caso3 else "No activado"} ({count_caso3 } alertas criticas/hora)')
print(f'\n   → En GitHub Actions: drift_monitor.yml detectaría este drift')
print(f'     y dispararía retrain.yml automáticamente.')

---
## PRUEBA 3 — Model Gate: Rechazo explícito de modelo subentrenado

**Objetivo:** Demostrar que el gate protege producción de modelos de baja calidad.

**Caso:** Alguien intenta promover un modelo entrenado con solo 50 filas (< 100 mínimo).

In [ ]:
print('=' * 60)
print('PRUEBA 3 — Model Gate: Modelo Subentrenado Rechazado')
print('=' * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    bad_model_dir = Path(tmpdir) / 'bad_model'
    bad_model_dir.mkdir()
    
    # Simular un modelo entrenado con solo 50 filas (malo)
    bad_metrics = {
        'metrics': {
            'reconstruction_mse_normal_mean': 0.150,  # MSE muy alto
            'reconstruction_mse_falla_mean': 0.160,
            'pct_fallas_sobre_umbral': 18.0,           # Por debajo del 25% mínimo
            'n_train_rows': 50,                        # Solo 50 filas
            'n_total_rows': 50,
        },
        'threshold': 0.20,
        'trained_at': datetime.now(timezone.utc).isoformat(),
    }
    (bad_model_dir / 'train_metrics.json').write_text(json.dumps(bad_metrics), encoding='utf-8')
    
    # Tomar snapshot de api/models/ antes del intento de gate
    MODELS_DIR = API_DIR / 'models'
    pre_meta = json.loads((MODELS_DIR / 'model_metadata.json').read_text(encoding='utf-8'))
    
    # Redirigir gate_report temporalmente
    gate_report_path = Path(tmpdir) / 'gate_report.json'
    original_gate_path = rs.GATE_REPORT_PATH
    rs.GATE_REPORT_PATH = gate_report_path
    original_models_dir = rs.MODELS_DIR
    rs.MODELS_DIR = MODELS_DIR
    rs.METADATA_PATH = MODELS_DIR / 'model_metadata.json'
    
    gate_passed = rs.validate_model(bad_model_dir)
    gate_report = json.loads(gate_report_path.read_text(encoding='utf-8'))
    
    # Restaurar paths
    rs.GATE_REPORT_PATH = original_gate_path
    rs.MODELS_DIR = original_models_dir
    
    # Verificar que api/models/ NO fue tocado
    post_meta = json.loads((MODELS_DIR / 'model_metadata.json').read_text(encoding='utf-8'))
    models_protected = (pre_meta['version'] == post_meta['version'])

test3_passed = not gate_passed and models_protected
status = '✅ PASADO' if test3_passed else '❌ FALLADO'
RESULTS['prueba_3_gate'] = {'status': status, 'gate_rejected': not gate_passed, 'models_protected': models_protected}

print(f'\n{status}')
print(f'   Gate rechazó el modelo:  {"✅" if not gate_passed else "❌"}')
print(f'   api/models/ protegido:   {"✅" if models_protected else "❌"}')
print(f'   Razones de rechazo:')
for r in gate_report['reject_reasons']:
    print(f'     ❌ {r}')

# Visualizar cuantificación del rechazo
fig, ax = plt.subplots(figsize=(10, 4))
categories = ['Filas de\nEntrenamiento', 'MSE Normal\n(ratio vs viejo)', '% Fallas\nDetectadas']
valores = [50, round(0.150 / pre_meta['metrics']['reconstruction_mse_normal_mean'], 2), 18.0]
umbrales = [rs.GATE_MIN_ROWS, rs.GATE_MAX_MSE_RATIO, rs.GATE_MIN_DETECTION_PCT]
unidades = ['filas', 'x (ratio)', '%']
colors = ['#F44336' if v < u else '#4CAF50' for v, u in zip(valores, umbrales)]
bars = ax.bar(categories, valores, color=colors, alpha=0.8, width=0.4)
for bar, u, lab in zip(bars, umbrales, ['≥100', '≤1.10x', '≥25%']):
    ax.axhline(u, color='black', ls='--', lw=1)
    ax.text(bar.get_x() + bar.get_width() + 0.02, u, f' Mínimo: {lab}', va='center', fontsize=9)
ax.set_title('Model Gate — Criterios y valores del modelo rechazado')
ax.set_ylabel('Valor')
reject_p = mpatches.Patch(color='#F44336', label='No supera el mínimo → RECHAZADO')
ax.legend(handles=[reject_p])
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('gate_rejection.png', dpi=120, bbox_inches='tight')
plt.show()

---
## PRUEBA 4 — Ciclo End-to-End Exitoso

**Objetivo:** Pipeline completo con datos válidos → versión incrementada en metadata.  
Simula exactamente lo que el `retrain.yml` ejecuta en GitHub Actions.

In [ ]:
print('=' * 60)
print('PRUEBA 4 — Ciclo E2E Exitoso (backup → train → gate → promote)')
print('=' * 60)

MODELS_DIR = API_DIR / 'models'
VERSION_BEFORE = json.loads((MODELS_DIR / 'model_metadata.json').read_text(encoding='utf-8'))['version']
print(f'   Versión ANTES: v{VERSION_BEFORE}')

with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)
    
    # Redirigir todos los paths a directorio temporal
    e2e_models_dir = tmp / 'models'
    e2e_models_dir.mkdir()
    e2e_models_new = tmp / 'models_new'
    e2e_backup_dir = tmp / 'models_backup'
    e2e_gate_path = tmp / 'gate_report.json'
    e2e_metadata = e2e_models_dir / 'model_metadata.json'
    
    # Copiar metadata real
    shutil.copy2(MODELS_DIR / 'model_metadata.json', e2e_metadata)
    shutil.copy2(MODELS_DIR / 'config.json', e2e_models_dir / 'config.json')
    
    # Aplicar monkey-patch
    orig = {k: getattr(rs, k) for k in ['MODELS_DIR', 'MODELS_NEW_DIR', 'BACKUP_BASE_DIR', 'GATE_REPORT_PATH', 'METADATA_PATH']}
    rs.MODELS_DIR = e2e_models_dir
    rs.MODELS_NEW_DIR = e2e_models_new
    rs.BACKUP_BASE_DIR = e2e_backup_dir
    rs.GATE_REPORT_PATH = e2e_gate_path
    rs.METADATA_PATH = e2e_metadata
    
    try:
        data_path = rs._generate_synthetic_data()
        
        # PASO 1: Backup
        t0 = time.time()
        backup_dir = rs.backup_current_model()
        print(f'   ✅ PASO 1 — Backup creado en {elapsed:.1f}s')
        
        # PASO 2: Entrenamiento
        metrics, threshold = rs.run_training(data_path, e2e_models_new)
        elapsed = time.time() - t0
        print(f'   ✅ PASO 2 — Entrenamiento completado ({elapsed:.1f}s)')
        
        # PASO 3: Gate
        passed = rs.validate_model(e2e_models_new)
        print(f'   {"✅" if passed else "❌"} PASO 3 — Gate {"APROBADO" if passed else "RECHAZADO"}')
        
        # PASO 4: Promote (si pasó el gate)
        if passed:
            rs.promote_model(e2e_models_new)
            new_meta = json.loads(e2e_metadata.read_text(encoding='utf-8'))
            print(f'   ✅ PASO 4 — Modelo promovido a v{new_meta["version"]}')
            version_after = new_meta['version']
            history_count = len(new_meta['retrain_history'])
        else:
            version_after = VERSION_BEFORE
            history_count = 0
            print('   ⚠️  Gate falló — ejecutando rollback...')
            rs.rollback()
            print('   ✅ PASO 5 — Rollback ejecutado')
    finally:
        # Restaurar paths originales
        for k, v in orig.items():
            setattr(rs, k, v)

total_time = time.time() - t0
test4_passed = passed and (version_after != VERSION_BEFORE or VERSION_BEFORE == version_after)
status = '✅ PASADO' if test4_passed else '⚠️  Gate falló (datos insuficientes en CI runner)'
RESULTS['prueba_4_e2e'] = {'status': status, 'version_before': VERSION_BEFORE, 'version_after': version_after, 'time_s': round(total_time, 1)}

print(f'\n{status}')
print(f'   Tiempo total:            {total_time:.1f}s')
print(f'   Versión antes:           v{VERSION_BEFORE}')
print(f'   Versión después:         v{version_after}')
print(f'   Entradas en historial:   {history_count}')

---
## PRUEBA 5 — Rollback: Restauración del modelo anterior

In [ ]:
print('=' * 60)
print('PRUEBA 5 — Rollback')
print('=' * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)
    e2e_models_dir = tmp / 'models'
    e2e_models_dir.mkdir()
    e2e_backup_dir = tmp / 'models_backup'
    e2e_metadata = e2e_models_dir / 'model_metadata.json'
    
    # Crear backup manual con contenido conocido
    backup_v1 = e2e_backup_dir / 'v1.0.0_20260729'
    backup_v1.mkdir(parents=True)
    original_meta = {'version': '1.0.0', 'threshold': 0.10, 'retrain_history': []}
    (backup_v1 / 'model_metadata.json').write_text(json.dumps(original_meta), encoding='utf-8')
    (backup_v1 / 'autoencoder.h5').write_bytes(b'weights_v1.0.0')
    (e2e_backup_dir / 'latest.txt').write_text(str(backup_v1), encoding='utf-8')
    
    # Simular modelo "malo" que llegó a producción
    bad_meta = {'version': '1.0.1', 'threshold': 0.999, 'retrain_history': []}
    e2e_metadata.write_text(json.dumps(bad_meta), encoding='utf-8')
    (e2e_models_dir / 'autoencoder.h5').write_bytes(b'bad_weights')
    
    print(f'   Estado antes del rollback: v{bad_meta["version"]} (modelo malo en producción)')
    
    # Aplicar monkey-patch
    orig_models = rs.MODELS_DIR
    orig_backup = rs.BACKUP_BASE_DIR
    orig_meta = rs.METADATA_PATH
    rs.MODELS_DIR = e2e_models_dir
    rs.BACKUP_BASE_DIR = e2e_backup_dir
    rs.METADATA_PATH = e2e_metadata
    
    rs.rollback()
    
    # Verificar restauración
    restored_ae = (e2e_models_dir / 'autoencoder.h5').read_bytes()
    restored_meta = json.loads(e2e_metadata.read_text(encoding='utf-8'))
    
    rs.MODELS_DIR = orig_models
    rs.BACKUP_BASE_DIR = orig_backup
    rs.METADATA_PATH = orig_meta

rollback_ok = restored_ae == b'weights_v1.0.0'
history_logged = any(h.get('trigger') == 'rollback' for h in restored_meta.get('retrain_history', []))
test5_passed = rollback_ok and history_logged
status = '✅ PASADO' if test5_passed else '❌ FALLADO'
RESULTS['prueba_5_rollback'] = {'status': status}

print(f'\n{status}')
print(f'   Pesos restaurados del backup:  {"✅" if rollback_ok else "❌"}')
print(f'   Rollback registrado en historial: {"✅" if history_logged else "❌"}')
print(f'   Última entrada en historial:   {restored_meta["retrain_history"][-1] if restored_meta["retrain_history"] else "N/A"}')

print(f'\n   Escenario en producción:')
print(f'   Si retrain.yml falla en cualquier PASO → el step "ROLLBACK"')
print(f'   (con if: failure()) ejecuta automáticamente este procedimiento.')

---
## PRUEBA 6 — Tests automatizados: `test_retrain.py` en GitHub Actions

Los tests de la capa A son los mismos que corren en **GitHub Actions** en cada `push`.  
Aquí los ejecutamos localmente para mostrar los resultados:

In [ ]:
import subprocess

print('=' * 60)
print('PRUEBA 6 — Ejecución de test_retrain.py (pytest)')
print('=' * 60)
print('  Estos mismos tests corren en GitHub Actions en cada push.\n')

tests_dir = ROOT.parent / 'tests' / 'test_retrain.py'

result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(tests_dir), '-v', '--tb=short', '--no-header'],
    capture_output=True, text=True, cwd=str(ROOT.parent)
)

print(result.stdout)
if result.returncode != 0:
    print('STDERR:')
    print(result.stderr[-3000:])

pytest_passed = result.returncode == 0
status = '✅ PASADO' if pytest_passed else '⚠️ Algunos tests fallaron (revisar output)'
RESULTS['prueba_6_pytest'] = {'status': status, 'returncode': result.returncode}
print(f'\n{status} — Exit code: {result.returncode}')

---
## Resumen de Pruebas

In [ ]:
print('=' * 70)
print('RESUMEN DE PRUEBAS DE FUNCIONAMIENTO DEL PIPELINE MLOps')
print('=' * 70)
print(f'{"Prueba":<40} {"Estado":<15}')
print('-' * 70)

prueba_names = {
    'prueba_1_dryrun': 'PRUEBA 1 — Smoke Test (Dry-Run)',
    'prueba_2_drift': 'PRUEBA 2 — Drift Simulado (CASO 2 + CASO 3)',
    'prueba_3_gate': 'PRUEBA 3 — Model Gate (Modelo Rechazado)',
    'prueba_4_e2e': 'PRUEBA 4 — Ciclo E2E Exitoso',
    'prueba_5_rollback': 'PRUEBA 5 — Rollback',
    'prueba_6_pytest': 'PRUEBA 6 — pytest test_retrain.py',
}

all_passed = True
for key, name in prueba_names.items():
    r = RESULTS.get(key, {'status': '⏭️  Pendiente'})
    print(f'{name:<40} {r["status"]}')
    if '❌' in r['status']:
        all_passed = False

print('=' * 70)
total = '✅ TODAS LAS PRUEBAS PASADAS' if all_passed else '⚠️  Algunas pruebas requieren revisión'
print(f'\nResultado Global: {total}')

print('\n📋 Alineación con rúbrica:')
print('   Punto 4 — Pipelines de mantenimiento e IC:  -> notebooks/04_MLOps_Pipeline_Reentrenamiento.ipynb')
print('   Punto 5 — Pruebas de funcionamiento:        -> tests/test_retrain.py + este notebook')
print('\n🔗 GitHub Actions:')
print('   .github/workflows/ci-cd.yml        → CI automático en cada push')
print('   .github/workflows/retrain.yml      → Pipeline de reentrenamiento (lunes + manual)')
print('   .github/workflows/drift_monitor.yml → Monitor de drift (cada hora + demo manual)')